<a href="https://colab.research.google.com/github/AnushaFreshStart/DL-agriculture-A/blob/main/Q7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 7

Task 1: Load the pre-trained CNN model in cnn_model variable using load_model() function and print model summary using summary() method.

In [ ]:
from tensorflow.keras.models import load_model

# Define the path using the model_name saved during training
keras_model_path = "best_model.keras"

# Load the pre-trained CNN model
cnn_model = load_model(keras_model_path)

# Inspect the architecture to find intermediate feature extraction layers
cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,441,541 (16.94 MB)

 Trainable params: 1,480,513 (5.65 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,961,028 (11.30 MB)

Task 2: Based on model.summary(), get the name of the layer from the CNN model for feature extraction in the variable feature_layer_name.

In [ ]:
# Layer name from the CNN model used for extracting feature maps
feature_layer_name = 'conv2d_3'

Task 3: Define the model architecture in a variable named hybrid_model using the build_cnn_vit_hybrid function.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_cnn_vit_hybrid(cnn_model, feature_layer_name, num_transformer_layers=4, num_heads=8, mlp_dim=2048, num_classes=2):
    """
    Constructs a hybrid CNN-ViT model by routing a symbolic input layer through the cnn_model.
    """
    # 1. Re-create a symbolic input tensor since cnn_model.input is uninstantiated
    input_tensor = layers.Input(shape=(64, 64, 3), name="input_image")

    # 2. Track and route the input through the sequential layers until we reach the target layer
    x = input_tensor
    intermediate_output = None

    for layer in cnn_model.layers:
        x = layer(x)
        if layer.name == feature_layer_name:
            intermediate_output = x
            break

    if intermediate_output is None:
        raise ValueError(f"Could not find layer {feature_layer_name} in the cnn_model.")

    # Shape of intermediate layer output: (Batch, H, W, C)
    _, h, w, c = intermediate_output.shape
    num_patches = h * w
    projection_dim = c  # Use channel dimension as token representation size

    # 3. Reshape spatial dimensions (H, W) into a sequence of patches (H*W, C)
    reshaped = layers.Reshape((num_patches, projection_dim))(intermediate_output)

    # 4. Add trainable Positional Embeddings to the projected patch tokens
    positions = tf.range(start=0, limit=num_patches, delta=1)
    position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)(positions)
    encoded = reshaped + position_embedding

    # 5. Multi-layer Transformer Encoder blocks
    for _ in range(num_transformer_layers):
        # Layer Normalization 1
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded)
        # Multi-Head Attention
        attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        # Skip connection 1
        x2 = layers.Add()([attention_output, encoded])

        # Layer Normalization 2
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP block
        mlp_output = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
        mlp_output = layers.Dense(projection_dim)(mlp_output)
        # Skip connection 2
        encoded = layers.Add()([mlp_output, x2])

    # 6. Global Average Pooling across patch tokens
    representation = layers.GlobalAveragePooling1D()(encoded)

    # 7. Classification Head (using softmax for 2 outputs as expected by categorical_crossentropy)
    logits = layers.Dense(num_classes, activation="softmax" if num_classes > 1 else "sigmoid")(representation)

    # Instantiate final Hybrid Model
    hybrid_model = Model(inputs=input_tensor, outputs=logits, name="CNN_ViT_Hybrid")
    return hybrid_model

# Construct the hybrid CNN-ViT model using 2 classes
hybrid_model = build_cnn_vit_hybrid(
    cnn_model=cnn_model,
    feature_layer_name=feature_layer_name,
    num_transformer_layers=4,
    num_heads=8,
    mlp_dim=2048,
    num_classes=2
)

# Print summary of the compiled hybrid architecture
hybrid_model.summary()

Model: "CNN_ViT_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 62, 62,    │        896 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 31, 31,    │          0 │ conv2d[13][0]     │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 29, 29,    │     18,496 │ max_pooling2d[12… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 14, 14,    │          0 │ conv2d_1[11][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 12, 12,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 6, 6, 128) │          0 │ conv2d_2[9][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 4, 4, 256) │    295,168 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 16, 256)   │          0 │ conv2d_3[7][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 16, 256)   │          0 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 256)   │  2,103,552 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 16, 256)   │          0 │ multi_head_atten… │
│                     │                   │            │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 16, 2048)  │    526,336 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 16, 256)   │    524,544 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 16, 256)   │          0 │ dense_15[0][0],   │
│                     │                   │            │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 13,010,754 (49.63 MB)

 Trainable params: 13,010,754 (49.63 MB)

 Non-trainable params: 0 (0.00 B)

Task 4: Compile the model hybrid_model.

In [ ]:
# Compile the hybrid model
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

Task 5: Define the training configuration of the hybrid_model.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

hybrid_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = hybrid_datagen.flow_from_directory(
    base_dir,
    target_size=(64, 64),
    batch_size=8,
    class_mode="categorical",
    subset="training",
    shuffle=True,
)

val_gen = hybrid_datagen.flow_from_directory(
    base_dir,
    target_size=(64, 64),
    batch_size=8,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

Found 4800 images belonging to 2 classes.
Found 1200 images belonging to 2 classes.


In [ ]:
# Train the hybrid model
fit = hybrid_model.fit(
    train_gen,
    epochs=3,
    steps_per_epoch=128,
    validation_data=val_gen,
    callbacks=[checkpoint_cb],
    verbose=1,
)

Epoch 1/3
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9583 - loss: 0.2931
Epoch 1: val_accuracy improved from None to 0.97417, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 260s 2s/step - accuracy: 0.9717 - loss: 0.2580 - val_accuracy: 0.9742 - val_loss: 0.1003
Epoch 2/3
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9846 - loss: 0.0968
Epoch 2: val_accuracy improved from 0.97417 to 0.98667, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 230s 2s/step - accuracy: 0.9893 - loss: 0.0679 - val_accuracy: 0.9867 - val_loss: 0.0594
Epoch 3/3
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9811 - loss: 0.0890
Epoch 3: val_accuracy improved from 0.98667 to 0.98917, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.9834 - loss: 0.0736 - val_accuracy: 